# Scheme Navigator — free IndicTrans2 pack generator
Run this on a **Colab GPU runtime**. Translation cache writes directly to Google Drive, so a Colab disconnect can resume instead of starting over.

**Before running:** request/accept access to `ai4bharat/indictrans2-en-indic-dist-200M` on Hugging Face, then create a **Read** access token from the same account. The token is entered into a hidden prompt and is never saved in this notebook/repository.

In Colab choose **Runtime → Change runtime type → T4 GPU**. This notebook keeps Colab's CUDA-enabled PyTorch, removes unused torchvision, pins `transformers==4.56.2`, and opts IndicTrans2 out of Hugging Face's newer dynamic-cache wrapper so the model can keep using its own legacy KV cache.


In [ ]:
import os
from google.colab import drive

# Never delete the repo while Colab is still inside it.
os.chdir('/content')
drive.mount('/content/drive', force_remount=False)
!rm -rf /content/scheme-navigator
!git clone https://github.com/um26/scheme-navigator.git /content/scheme-navigator
os.chdir('/content/scheme-navigator')
print('✅ Working directory:', os.getcwd())


In [ ]:
import os
os.chdir('/content/scheme-navigator')

# Keep Colab's CUDA-enabled torch. IndicTrans2 is text-only, so torchvision is unnecessary.
!pip -q uninstall -y torchvision >/dev/null 2>&1 || true
!pip -q install -U 'transformers==4.56.2' indictranstoolkit sentencepiece sacremoses accelerate huggingface_hub
!npm install --silent

import torch, transformers
from packaging.version import Version
print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('CUDA available:', torch.cuda.is_available())
if Version(torch.__version__.split('+')[0]) < Version('2.5'):
    raise RuntimeError('IndicTransToolkit needs torch>=2.5. Start a fresh Colab GPU runtime; do not install a CPU-only torch build.')
print('✅ Compatible dependencies installed')


## Authenticate to Hugging Face
The web page saying **granted access** is necessary, but Colab must also authenticate using a token from that same account. Paste a Hugging Face **Read** token into the hidden prompt. Do not paste the token into chat, notebook text, screenshots, or GitHub.


In [ ]:
import os
from getpass import getpass
from huggingface_hub import HfApi

os.chdir('/content/scheme-navigator')
MODEL_NAME = 'ai4bharat/indictrans2-en-indic-dist-200M'
HF_TOKEN = getpass('Paste your Hugging Face READ token (input is hidden): ')
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

api = HfApi(token=HF_TOKEN)
me = api.whoami()
print(f"✅ Authenticated to Hugging Face as: {me.get('name') or me.get('fullname') or 'your account'}")
try:
    info = api.model_info(MODEL_NAME, token=HF_TOKEN)
    print(f"✅ Gated model access verified: {info.modelId}")
except Exception as exc:
    raise RuntimeError(
        'Hugging Face login worked, but this token/account still cannot access the IndicTrans2 model. '
        'Make sure the token was created from the SAME account that shows granted access, then rerun this cell.'
    ) from exc


In [ ]:
import os
from pathlib import Path

os.chdir('/content/scheme-navigator')
assert Path('scripts/translate-catalog.py').exists(), 'Repo setup is incomplete. Rerun the first setup cell.'

# Put the append-only translation cache DIRECTLY on Drive. Every completed batch
# is therefore persistent even if the free Colab runtime disconnects.
drive_root = Path('/content/drive/MyDrive/scheme-navigator-i18n')
drive_cache = drive_root / 'cache'
drive_cache.mkdir(parents=True, exist_ok=True)
Path('.translation-work').mkdir(exist_ok=True)
local_cache = Path('.translation-work/cache')
if local_cache.is_symlink() or local_cache.exists():
    if local_cache.is_symlink():
        local_cache.unlink()
    else:
        import shutil
        for p in local_cache.glob('*.jsonl'):
            target = drive_cache / p.name
            with target.open('a', encoding='utf-8') as out, p.open('r', encoding='utf-8') as src:
                out.write(src.read())
        shutil.rmtree(local_cache)
local_cache.symlink_to(drive_cache, target_is_directory=True)

!mkdir -p public/i18n/schemes
!cp /content/drive/MyDrive/scheme-navigator-i18n/generated-ui.json public/i18n/generated-ui.json 2>/dev/null || true
!cp /content/drive/MyDrive/scheme-navigator-i18n/generated.js lib/i18n/generated.js 2>/dev/null || true
!cp /content/drive/MyDrive/scheme-navigator-i18n/*.json.gz public/i18n/schemes/ 2>/dev/null || true
print('✅ Persistent cache:', local_cache.resolve())
print('✅ Repo/cache ready at', os.getcwd())


In [ ]:
import os, runpy, sys
from pathlib import Path
import torch, transformers
from transformers.generation.utils import GenerationMixin

os.chdir('/content/scheme-navigator')
assert Path('scripts/translate-catalog.py').exists(), 'Translation script missing. Rerun the setup cell.'
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not enabled. In Colab choose Runtime → Change runtime type → T4 GPU, then restart and rerun from the top.')
print('✅ GPU:', torch.cuda.get_device_name(0))
print('✅ Transformers:', transformers.__version__)

# IndicTrans2 manages a legacy tuple-of-tuples KV cache. Transformers 4.56 can
# otherwise wrap it in EncoderDecoderCache, producing None entries in generation.
# This is the same compatibility escape hatch used by the upstream fix.
GenerationMixin._supports_default_dynamic_cache = classmethod(lambda cls: False)
print('✅ IndicTrans2 legacy KV-cache compatibility enabled')

LOCALES = 'hi'
sys.argv = [
    '/content/scheme-navigator/scripts/translate-catalog.py',
    '--locales', LOCALES,
    '--batch-size', '32',
]
runpy.run_path('/content/scheme-navigator/scripts/translate-catalog.py', run_name='__main__')


In [ ]:
import os
from pathlib import Path

os.chdir('/content/scheme-navigator')
packs = list(Path('public/i18n/schemes').glob('*.json.gz'))
if not packs:
    raise RuntimeError('No translation packs exist yet. Run the translation cell successfully before packaging.')

!mkdir -p /content/drive/MyDrive/scheme-navigator-i18n
!cp public/i18n/generated-ui.json /content/drive/MyDrive/scheme-navigator-i18n/generated-ui.json
!cp lib/i18n/generated.js /content/drive/MyDrive/scheme-navigator-i18n/generated.js
!cp public/i18n/schemes/*.json.gz /content/drive/MyDrive/scheme-navigator-i18n/
!rm -f /content/scheme-navigator-translations.zip
!zip -q -r /content/scheme-navigator-translations.zip public/i18n lib/i18n/generated.js
print('✅ Translation packs:', ', '.join(p.name for p in packs))
print('Artifact: /content/scheme-navigator-translations.zip')
